# v10 pilot — Qwen3-VL vs Florence-2 trên 100 sản phẩm đã kiểm tay

Mục tiêu: xem caption của Qwen3-VL có **bổ sung thông tin thị giác chưa có trong title** hay không, trước khi tạo lại toàn bộ corpus 108,753 item.

Ba điều kiện chạy cho mỗi sản phẩm:

| Điều kiện | Đầu vào | Mục đích |
|---|---|---|
| `structured` | ảnh + title | Prompt chính: chỉ tả điều **chưa có trong title**, cấm chép chữ trên ảnh, báo `MISMATCH` nếu ảnh không khớp |
| `generic` | chỉ ảnh | Đối chứng kiểu Florence: "mô tả ảnh trong 1 câu" |
| `title_only` | chỉ title, **không ảnh** | Đo phần "kiến thức sẵn có": model đoán thuộc tính chỉ từ title. Nếu `structured` ≈ `title_only` thì thông tin không đến từ ảnh |

Mẫu 100 item cố định (seed `20260924`, 50 Video_Games + 10 mỗi miền khác), kèm caption Florence-2 và nhãn thủ công cũ để so sánh.

**Ngưỡng quyết định (chốt trước khi chạy):** trên 100 item, gán nhãn lại cho `structured`:
ADDS ≥ 40, OCR/lặp title ≤ 20, SUSPECT (không tính MISMATCH đúng) ≤ 10. Không đạt → đóng nhánh caption.

Chọn runtime GPU: **A100 hoặc L4** chạy được 8B bf16; **T4** tự chuyển sang 4B fp16.

In [ ]:
!pip -q install "transformers>=4.57.0,<5" pillow requests pandas
# Không dùng -U cho accelerate/torch: nâng cấp có thể kéo torch bản CUDA 13 lẫn với thư viện CUDA 12 của Colab
# và gây lỗi "failed to open libnvrtc-builtins.so.13.0". Nếu đã lỡ nâng cấp: Runtime → Disconnect and delete runtime rồi chạy lại.

In [ ]:
import json, os, re, time, hashlib, io
from pathlib import Path
import requests, torch, pandas as pd
from PIL import Image

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
cc = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
# T4 (7.5) chỉ giả lập bf16; bf16 trên T4 đi qua kernel biên dịch lúc chạy (NVRTC) → lỗi libnvrtc. Chỉ dùng bf16 từ Ampere (8.0) trở lên.
bf16_ok = cc >= (8, 0)
# Có thể ép model bằng cách đặt MODEL_ID thủ công.
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct" if vram_gb >= 30 else "Qwen/Qwen3-VL-4B-Instruct"
DTYPE = torch.bfloat16 if bf16_ok else torch.float16
MAX_NEW_TOKENS = 96
OUT = Path("/content/v10_qwen3vl_pilot"); OUT.mkdir(parents=True, exist_ok=True)
IMG_DIR = OUT / "images"; IMG_DIR.mkdir(exist_ok=True)
torch.manual_seed(20260924)
import transformers
print(gpu, f"{vram_gb:.1f} GB", "cc", cc, MODEL_ID, DTYPE)
print("torch", torch.__version__, "cuda", torch.version.cuda, "| transformers", transformers.__version__)

# Tự kiểm tra GPU ngay: nếu môi trường CUDA hỏng thì báo lỗi ở đây, trước khi tải model.
x = torch.arange(1, 9, device="cuda", dtype=DTYPE)
print("GPU self-test ok:", float(x.prod()), float(x.float().exp2().sum()))

In [ ]:
ITEMS = json.loads(r'''[{"n": 1, "global_id": 66332, "domain": "Video_Games", "asin": "B001NJKHWI", "title": "Valkyrie Profile: Covenant of the Plume - Nintendo DS", "image_url": "https://m.media-amazon.com/images/I/81fLF2Gfb1L._SL1500_.jpg", "florence_caption": "valkyrie profile covenant of the plume", "florence_label": "OCR"}, {"n": 2, "global_id": 75036, "domain": "Video_Games", "asin": "B00RSXRLUE", "title": "Mayflash GameCube Controller Adapter for Wii U, PC USB and Switch, 4 Port", "image_url": "https://m.media-amazon.com/images/I/71sdpOIRLZL._AC_SL1500_.jpg", "florence_caption": "A black controller adapter with a usb cable connected to it.", "florence_label": "ADDS"}, {"n": 3, "global_id": 68461, "domain": "Video_Games", "asin": "B004Y6DPQM", "title": "Dance Dance Revolution WII", "image_url": "https://m.media-amazon.com/images/I/914I7d20YjL._SL1500_.jpg", "florence_caption": "dance dance revolution wii", "florence_label": "OCR"}, {"n": 4, "global_id": 75237, "domain": "Video_Games", "asin": "B0000DJX7I", "title": "Grand Theft Auto Double Pack: Grand Theft Auto III / Grand Theft Auto: Vice City", "image_url": "https://m.media-amazon.com/images/I/512AW0XX1RL.jpg", "florence_caption": "grand theft auto rockstar games double pack", "florence_label": "OCR"}, {"n": 5, "global_id": 75400, "domain": "Video_Games", "asin": "B000HCQK0A", "title": "Crackdown - Xbox 360", "image_url": "https://m.media-amazon.com/images/I/61kZ4VKDO-L.jpg", "florence_caption": "crackdown xbox 360 game", "florence_label": "OCR"}, {"n": 6, "global_id": 67018, "domain": "Video_Games", "asin": "B000KA5T6A", "title": "Microsoft Xbox 360 Wireless Controller for Windows", "image_url": "https://m.media-amazon.com/images/I/41jHIlQtUNL._AC_.jpg", "florence_caption": "a white xbox 360 controller", "florence_label": "ADDS"}, {"n": 7, "global_id": 73073, "domain": "Video_Games", "asin": "B08BRKGLT8", "title": "Armor3 \"ReadyVolt\" USB AC Adapter for Turbografx-16 Mini and PC Engine Mini - Not Machine Specific", "image_url": "https://m.media-amazon.com/images/I/71wi9SHb2FL._SL1500_.jpg", "florence_caption": "A black box with a black charger inside of it.", "florence_label": "ADDS"}, {"n": 8, "global_id": 70638, "domain": "Video_Games", "asin": "B00005B4B5", "title": "Necronomicon - PC", "image_url": "https://m.media-amazon.com/images/I/516XV4N4RRL.jpg", "florence_caption": "necronomicon in the darkness of horror", "florence_label": "OCR"}, {"n": 9, "global_id": 72106, "domain": "Video_Games", "asin": "B0050SVMGS", "title": "Amazon Basics 3-Pack Stylus for Nintendo 3DS (Officially Licensed by Nintendo)", "image_url": "https://m.media-amazon.com/images/I/71cQL6rrEcL._SL1500_.jpg", "florence_caption": "A set of three pens sitting next to each other on a white surface.", "florence_label": "REDUNDANT"}, {"n": 10, "global_id": 73340, "domain": "Video_Games", "asin": "B00AZNBP1A", "title": "Final Fantasy XI: Seekers of Adoulin [Download]", "image_url": "https://m.media-amazon.com/images/I/71QW410d-YL._SL1500_.jpg", "florence_caption": "final fantasy xii seekers of adoulin", "florence_label": "OCR"}, {"n": 11, "global_id": 74912, "domain": "Video_Games", "asin": "B001EM7TUC", "title": "SAS Secure Tomorrow - PC", "image_url": "https://m.media-amazon.com/images/I/51XURCwZ8oL.jpg", "florence_caption": "sas secure tomorrow pc full espaol", "florence_label": "OCR"}, {"n": 12, "global_id": 71539, "domain": "Video_Games", "asin": "B000EG8RDC", "title": "eForCity 3 Pack of Plastic Stylus (Stylo Styli Styrograph) for Nintendo DS (handheld video game system)", "image_url": "https://m.media-amazon.com/images/I/412RX3qoz+L.jpg", "florence_caption": "A set of three gray plastic pens on a white background.", "florence_label": "ADDS"}, {"n": 13, "global_id": 71737, "domain": "Video_Games", "asin": "B00001NTSO", "title": "Sid Meier's Alpha Centauri - PC", "image_url": "https://m.media-amazon.com/images/I/712FMGNX74L.gif", "florence_caption": "The cover of Sid Meier's Alpha Centauri: The Future of Mankind.", "florence_label": "OCR"}, {"n": 14, "global_id": 73316, "domain": "Video_Games", "asin": "B0000AHRPM", "title": "SpongeBob SquarePants: The Battle for Bikini Bottom", "image_url": "https://m.media-amazon.com/images/I/51KWSHC52ML.jpg", "florence_caption": "spongebob squarepants battle for bikini bottom", "florence_label": "OCR"}, {"n": 15, "global_id": 74939, "domain": "Video_Games", "asin": "B07HYCZ68N", "title": "Suncala 128MB Memory Card for Playstation 2, High Speed Memory Card for Sony PS2", "image_url": "https://m.media-amazon.com/images/I/71Pyrq-7tfL._AC_SL1500_.jpg", "florence_caption": "suncala 128mb memory card for ps2", "florence_label": "OCR"}, {"n": 16, "global_id": 70083, "domain": "Video_Games", "asin": "B0002MPT6Q", "title": "Rome: Total War - PC", "image_url": "https://m.media-amazon.com/images/I/41G4ZKJ57CL.jpg", "florence_caption": "rome total war", "florence_label": "OCR"}, {"n": 17, "global_id": 67037, "domain": "Video_Games", "asin": "B00H51BJBG", "title": "Deception IV: Blood Ties - PlayStation Vita", "image_url": "https://m.media-amazon.com/images/I/81P6A6l34QL._SL1478_.jpg", "florence_caption": "deception iv blood ties", "florence_label": "OCR"}, {"n": 18, "global_id": 70402, "domain": "Video_Games", "asin": "B072K2DCFM", "title": "Hyperkin HD Cable for Wii", "image_url": "https://m.media-amazon.com/images/I/71xOiwoAQlL._AC_SL1500_.jpg", "florence_caption": "A usb cable connected to a micro USB adapter.", "florence_label": "SUSPECT"}, {"n": 19, "global_id": 69458, "domain": "Video_Games", "asin": "B00000K1XU", "title": "Shadow Man", "image_url": "https://m.media-amazon.com/images/I/51KumfXnVuL._SL1500_.jpg", "florence_caption": "A pink humidifier with a white background.", "florence_label": "SUSPECT"}, {"n": 20, "global_id": 70435, "domain": "Video_Games", "asin": "B00PLQKFZ8", "title": "Sony Playstation: Playstation Console Shaped Messenger Bag", "image_url": "https://m.media-amazon.com/images/I/31k4-O2MVLL.jpg", "florence_caption": "a white bag with a playstation logo on it", "florence_label": "ADDS"}, {"n": 21, "global_id": 66673, "domain": "Video_Games", "asin": "B00DE2W4PK", "title": "PlayStation 4 Killzone Launch Day Bundle", "image_url": "https://m.media-amazon.com/images/I/41nkwHrns7L.jpg", "florence_caption": "a black playstation 4 console with a game controller next to it", "florence_label": "REDUNDANT"}, {"n": 22, "global_id": 67499, "domain": "Video_Games", "asin": "B001DT02JG", "title": "Microsoft Xbox 360 Game System HDMI Console 60GB", "image_url": "https://m.media-amazon.com/images/I/41tO08f3zzL.jpg", "florence_caption": "a white xbox 360 with a controller next to it", "florence_label": "ADDS"}, {"n": 23, "global_id": 72136, "domain": "Video_Games", "asin": "B0009A4ETE", "title": "Fullmetal Alchemist 2: Curse of the Crimson Elixir - PlayStation 2", "image_url": "https://m.media-amazon.com/images/I/51VH0PXTQ7L.jpg", "florence_caption": "fullmetal alchemist 2 curse of the crimson elixir", "florence_label": "OCR"}, {"n": 24, "global_id": 73788, "domain": "Video_Games", "asin": "B00K0QSQ0U", "title": "Nintendo 2DS-White/Pink with Bonus Disney Magical World Carrying Case (Used)", "image_url": "https://m.media-amazon.com/images/I/71fbzsR-QcL._SL1500_.jpg", "florence_caption": "a nintendo 3ds with a mickey mouse case next to it", "florence_label": "REDUNDANT"}, {"n": 25, "global_id": 71399, "domain": "Video_Games", "asin": "B00004LN2T", "title": "WWF Smackdown!", "image_url": "https://m.media-amazon.com/images/I/71dGLkUUYKL._SL1105_.jpg", "florence_caption": "The cover of the game WWE SmackDown!", "florence_label": "OCR"}, {"n": 26, "global_id": 67319, "domain": "Video_Games", "asin": "B00005KAPT", "title": "Chu Chu Rocket", "image_url": "https://m.media-amazon.com/images/I/811sxQY2GOL._SL1500_.jpg", "florence_caption": "A game boy advance and a game boy memory card on a table.", "florence_label": "GENERIC"}, {"n": 27, "global_id": 70261, "domain": "Video_Games", "asin": "B000067FDX", "title": "Shadowbane - PC/Mac", "image_url": "https://m.media-amazon.com/images/I/51EPK0KZ1HL.jpg", "florence_caption": "shadowbate the next great online rpg", "florence_label": "OCR"}, {"n": 28, "global_id": 73681, "domain": "Video_Games", "asin": "B00007KUW7", "title": "Rayman 3: Hoodlum Havoc", "image_url": "https://m.media-amazon.com/images/I/51VS1CJYH4L.jpg", "florence_caption": "rayman 3 hoodlum havoc", "florence_label": "OCR"}, {"n": 29, "global_id": 69684, "domain": "Video_Games", "asin": "B01M5GG9I7", "title": "Microsoft CWT-00001 Xbox Controller + Wireless Adapter for Windows", "image_url": "https://m.media-amazon.com/images/I/51cztbu0zrL._AC_SL1200_.jpg", "florence_caption": "A black Xbox controller next to a USB flash drive.", "florence_label": "REDUNDANT"}, {"n": 30, "global_id": 74311, "domain": "Video_Games", "asin": "B000006OVJ", "title": "Mega Man Legends - PlayStation", "image_url": "https://m.media-amazon.com/images/I/91r1e8JXvaL._SL1500_.jpg", "florence_caption": "A video game case sitting on top of a table.", "florence_label": "GENERIC"}, {"n": 31, "global_id": 74174, "domain": "Video_Games", "asin": "B01J4JTZ0S", "title": "Ologymart Two 2 Controllers Bundle Compatible with Super Nintendo SNES Bulk Packaging Pack Of 2", "image_url": "https://m.media-amazon.com/images/I/41SDseDBRVL._AC_SL1500_.jpg", "florence_caption": "a pair of nintendo classic mini controllers", "florence_label": "REDUNDANT"}, {"n": 32, "global_id": 73610, "domain": "Video_Games", "asin": "B00001X50M", "title": "Metal Gear Solid", "image_url": "https://m.media-amazon.com/images/I/71XbLFYlW9L._SL1500_.jpg", "florence_caption": "metal gear solid playstation", "florence_label": "OCR"}, {"n": 33, "global_id": 73499, "domain": "Video_Games", "asin": "B004WMM2NA", "title": "Animal Crossing: City Folk (Nintendo Selects)", "image_url": "https://m.media-amazon.com/images/I/811cT4hA-8L._SL1500_.jpg", "florence_caption": "animal crossing city folk nintendo selects", "florence_label": "OCR"}, {"n": 34, "global_id": 73173, "domain": "Video_Games", "asin": "B00005T7ZN", "title": "Shadow Hearts - PlayStation 2", "image_url": "https://m.media-amazon.com/images/I/61vomi7hYGL._SL1000_.jpg", "florence_caption": "shadow hearts playstation 2", "florence_label": "OCR"}, {"n": 35, "global_id": 73394, "domain": "Video_Games", "asin": "B002WSR8BC", "title": "Deadly Premonition - Xbox 360", "image_url": "https://m.media-amazon.com/images/I/51Trz9LTYeL.jpg", "florence_caption": "deadly premonition xbox 360", "florence_label": "OCR"}, {"n": 36, "global_id": 70620, "domain": "Video_Games", "asin": "B000038A7C", "title": "Tom Clancy's Rainbow Six", "image_url": "https://m.media-amazon.com/images/I/91W78XpAt5L._SL1500_.jpg", "florence_caption": "tom clancy's rainbow six n64", "florence_label": "ADDS"}, {"n": 37, "global_id": 66716, "domain": "Video_Games", "asin": "B0002VK8YA", "title": "E.T. the Extra-Terrestrial", "image_url": "https://m.media-amazon.com/images/I/71GwCody8TL._SL1500_.jpg", "florence_caption": "A video game cartridge with a picture of a man and a dog.", "florence_label": "ADDS"}, {"n": 38, "global_id": 75229, "domain": "Video_Games", "asin": "B000K9WNIS", "title": "Dungeons & Dragons Tactics - Sony PSP", "image_url": "https://m.media-amazon.com/images/I/91nK9UfNQGL._SL1500_.jpg", "florence_caption": "Two video games sitting next to each other on a table.", "florence_label": "GENERIC"}, {"n": 39, "global_id": 68814, "domain": "Video_Games", "asin": "B00004SVUK", "title": "The Simpsons: Bart vs. the World", "image_url": "https://m.media-amazon.com/images/I/A1KmHQ5H+LL._SL1500_.jpg", "florence_caption": "The simpsons bart vs the world nintendo entertainment system.", "florence_label": "ADDS"}, {"n": 40, "global_id": 75082, "domain": "Video_Games", "asin": "B0000A03C3", "title": "Conflict: Desert Storm 2 Back to Baghdad - Xbox", "image_url": "https://m.media-amazon.com/images/I/91TH-i6ZqVL._SL1500_.jpg", "florence_caption": "conflict desert storm ii back to baghdad", "florence_label": "OCR"}, {"n": 41, "global_id": 72514, "domain": "Video_Games", "asin": "B00QCMDO44", "title": "Divinity: Original Sin - Multiple (Windows and Mac): select platform(s) Standard Edition", "image_url": "https://m.media-amazon.com/images/I/81M61AGLUqL._SL1500_.jpg", "florence_caption": "divinity original sin pc", "florence_label": "OCR"}, {"n": 42, "global_id": 72838, "domain": "Video_Games", "asin": "B0083LPK58", "title": "Max Payne 3", "image_url": "https://m.media-amazon.com/images/I/91MSgIs-OQL._SL1500_.jpg", "florence_caption": "A video game case with a video game on it.", "florence_label": "GENERIC"}, {"n": 43, "global_id": 68015, "domain": "Video_Games", "asin": "B0002ILS1K", "title": "Paper Mario: The Thousand-Year Door", "image_url": "https://m.media-amazon.com/images/I/61Q05FCGB9L.jpg", "florence_caption": "paper mario the thousand year door", "florence_label": "OCR"}, {"n": 44, "global_id": 74338, "domain": "Video_Games", "asin": "B005QA98JS", "title": "Twisted Lands: Insomniac - Collector's Edition Bonus Pack", "image_url": "https://m.media-amazon.com/images/I/91iMoFdPfGL._AC_SL1500_.jpg", "florence_caption": "Twisted Lands: Insomniac Collector's Edition", "florence_label": "OCR"}, {"n": 45, "global_id": 70562, "domain": "Video_Games", "asin": "B0001HAI9U", "title": "SVC Chaos - SNK vs Capcom", "image_url": "https://m.media-amazon.com/images/I/515GKW0ZH0L.jpg", "florence_caption": "the king of fighters xiii", "florence_label": "SUSPECT"}, {"n": 46, "global_id": 68907, "domain": "Video_Games", "asin": "B005GMOJQI", "title": "O-Games 209635 Jewel Time Deluxe -Nintendo DS", "image_url": "https://m.media-amazon.com/images/I/61sgGGECMkL.jpg", "florence_caption": "more ways to play jewel time deluxe", "florence_label": "OCR"}, {"n": 47, "global_id": 73553, "domain": "Video_Games", "asin": "B00008DHNJ", "title": "Pitfall: The Lost Expedition - PlayStation 2", "image_url": "https://m.media-amazon.com/images/I/619gxmXq3QL._SL1140_.jpg", "florence_caption": "pitfall the lost expedition", "florence_label": "OCR"}, {"n": 48, "global_id": 74541, "domain": "Video_Games", "asin": "B000II94KQ", "title": "Castlevania: Legacy of Darkness", "image_url": "https://m.media-amazon.com/images/I/41GAFEZ6JPL.jpg", "florence_caption": "castlevania legacy of darkness", "florence_label": "OCR"}, {"n": 49, "global_id": 75520, "domain": "Video_Games", "asin": "B003IU1AHQ", "title": "I Spy Universe - Nintendo DS", "image_url": "https://m.media-amazon.com/images/I/817rnFfKKRL._SL1500_.jpg", "florence_caption": "The cover of the game I Spy Universe.", "florence_label": "OCR"}, {"n": 50, "global_id": 72973, "domain": "Video_Games", "asin": "B00029QOQS", "title": "Rollercoaster Tycoon 3 - PC", "image_url": "https://m.media-amazon.com/images/I/51ZZM079Y3L.jpg", "florence_caption": "roller coaster tycoon 3", "florence_label": "OCR"}, {"n": 51, "global_id": 8321, "domain": "Arts_Crafts_and_Sewing", "asin": "B00G8D46NA", "title": "Spectrum Noir 96-pen SET Brights Lights Pastels Darks Next Generation Alcohol Ink Markers Pens Refillable", "image_url": "https://m.media-amazon.com/images/I/51Tw1YEqamL._AC_.jpg", "florence_caption": "spectrum noir permanent markers - set of 4", "florence_label": "SUSPECT"}, {"n": 52, "global_id": 3930, "domain": "Arts_Crafts_and_Sewing", "asin": "B000W5HTWA", "title": "DMC U1541 Embroidery Tracing Paper, Yellow/Blue, 2-Pack", "image_url": "https://m.media-amazon.com/images/I/71kdvbatJVL._AC_SL1200_.jpg", "florence_caption": "dmc embroidery tracing paper", "florence_label": "OCR"}, {"n": 53, "global_id": 1932, "domain": "Arts_Crafts_and_Sewing", "asin": "B000SN4ZXM", "title": "Turquoise Gem Round Beads 4mm / 16 Inch Strand", "image_url": "https://m.media-amazon.com/images/I/61a77tttfmL._AC_SL1000_.jpg", "florence_caption": "a strand of turquoise glass beads on a white background", "florence_label": "REDUNDANT"}, {"n": 54, "global_id": 12055, "domain": "Arts_Crafts_and_Sewing", "asin": "B007F0UV02", "title": "The Crafters Workshop TCW-252 Template, 12 by 12-Inch, Aspen Trees", "image_url": "https://m.media-amazon.com/images/I/71YXY1Il2UL._AC_SL1200_.jpg", "florence_caption": "a black and white image of birch trees", "florence_label": "REDUNDANT"}, {"n": 55, "global_id": 368, "domain": "Arts_Crafts_and_Sewing", "asin": "1937193284", "title": "Jaybird Night Sky Quilt, Multi Colored", "image_url": "https://m.media-amazon.com/images/I/71YsHXXFr7L._AC_SL1202_.jpg", "florence_caption": "a colorful quilt on a wooden fence", "florence_label": "REDUNDANT"}, {"n": 56, "global_id": 7988, "domain": "Arts_Crafts_and_Sewing", "asin": "B011T5V08K", "title": "ArtResin - Epoxy Resin - Clear - Non-Toxic - 1 gal (0.5 gal Resin + 0.5 gal Hardener) (3.78L)", "image_url": "https://m.media-amazon.com/images/I/81KlNhVVJLL._AC_SL1500_.jpg", "florence_caption": "A gallon of art resin next to a gallon of hardener.", "florence_label": "REDUNDANT"}, {"n": 57, "global_id": 3650, "domain": "Arts_Crafts_and_Sewing", "asin": "B011Q0BEZM", "title": "American Crafts 340272 Sticky Thumb Foam DT ASST, White Dots, Assorted Sizes", "image_url": "https://m.media-amazon.com/images/I/81hKxC8w2UL._AC_SL1500_.jpg", "florence_caption": "sticky thumb adhesive foam dots", "florence_label": "OCR"}, {"n": 58, "global_id": 9693, "domain": "Arts_Crafts_and_Sewing", "asin": "B01FEWF10W", "title": "Sizzix, Multi Color, Embossing Folder , Snowfall Speckles by Tim Holtz, One Size", "image_url": "https://m.media-amazon.com/images/I/71jUcUKw3QL._AC_SL1200_.jpg", "florence_caption": "sizzix texture fades embossing folder", "florence_label": "OCR"}, {"n": 59, "global_id": 4782, "domain": "Arts_Crafts_and_Sewing", "asin": "B010MT6OQM", "title": "Aurifil Thread Set Classic Collection 50wt Cotton 12 Large (1422 yard) Spools", "image_url": "https://m.media-amazon.com/images/I/71mRz6vHaQL._AC_SL1024_.jpg", "florence_caption": "aurifil cotton thread set", "florence_label": "OCR"}, {"n": 60, "global_id": 6799, "domain": "Arts_Crafts_and_Sewing", "asin": "B07H2D1SFP", "title": "Mandala Dotting Tools for Painting Rocks Mandala Painting Dotting Stencil Dot Mandala Kit 33PCS", "image_url": "https://m.media-amazon.com/images/I/71PVug+LW0S._AC_SL1500_.jpg", "florence_caption": "a bunch of different colored pens sitting next to each other", "florence_label": "GENERIC"}, {"n": 61, "global_id": 15914, "domain": "Electronics", "asin": "B002JYCIR8", "title": "Fotodiox Pro Lens Mount Adapter Compatible with Pentax 6x7 Lenses to Pentax K-Mount Cameras", "image_url": "https://m.media-amazon.com/images/I/81Fj1qaRXwL._AC_SL1500_.jpg", "florence_caption": "a close up of a camera lens adapter on a white background", "florence_label": "REDUNDANT"}, {"n": 62, "global_id": 16634, "domain": "Electronics", "asin": "B001CJOLBW", "title": "Monoprice VGA to RCA Adapter PC to TV Video Converter- Blue", "image_url": "https://m.media-amazon.com/images/I/41TYz-adklL._AC_.jpg", "florence_caption": "vga to vga video converter", "florence_label": "SUSPECT"}, {"n": 63, "global_id": 26982, "domain": "Electronics", "asin": "B016YKB8ZU", "title": "Audio-Technica ATH-M40x Renewed", "image_url": "https://m.media-amazon.com/images/I/816xGwuaaOL._AC_SL1500_.jpg", "florence_caption": "A pair of headphones sitting on top of each other.", "florence_label": "REDUNDANT"}, {"n": 64, "global_id": 16984, "domain": "Electronics", "asin": "B093H7GDRH", "title": "MasiBloom Silicone Case for AirTag 2021 Holder Accessories with Keychain Ring Protective Skin Cover (Turquoise Blue, for AirTag)", "image_url": "https://m.media-amazon.com/images/I/41ZGgTsSN9S._AC_SL1000_.jpg", "florence_caption": "a green keychain with a metal ring on it", "florence_label": "REDUNDANT"}, {"n": 65, "global_id": 13252, "domain": "Electronics", "asin": "B00CCVKQRM", "title": "Importer520 New Gold Plated Hdmi Female to Dvi-d Male Video Adaptor", "image_url": "https://m.media-amazon.com/images/I/51WgeOckpaL._AC_SL1278_.jpg", "florence_caption": "A pair of white earphones with a charging cable.", "florence_label": "SUSPECT"}, {"n": 66, "global_id": 12899, "domain": "Electronics", "asin": "B01E06JSI4", "title": "Planar Helium PCT2235 Touch Screen 22\" LED LCD Full HD Resolution Monitor with Helium Stand,black", "image_url": "https://m.media-amazon.com/images/I/71qMHAfAYHL._AC_SL1500_.jpg", "florence_caption": "A smart watch is displayed on a large screen.", "florence_label": "GENERIC"}, {"n": 67, "global_id": 22633, "domain": "Electronics", "asin": "B0765BD4PF", "title": "Samsung UE510 LED Display Monitor, Black, 28\" 4K (Renewed)", "image_url": "https://m.media-amazon.com/images/I/81xmZQQBFBL._AC_SL1500_.jpg", "florence_caption": "A picture of a city skyline at night on a computer monitor.", "florence_label": "GENERIC"}, {"n": 68, "global_id": 25832, "domain": "Electronics", "asin": "B01EGCQ4WU", "title": "Transcend USB 3.1/3.0 Super Speed Type-C Multi-Card Reader for SDHC/SDXC/MS/CF (TS-RDC8K)", "image_url": "https://m.media-amazon.com/images/I/61F9raU5osL._AC_SL1371_.jpg", "florence_caption": "A black USB 3.1/3.0 card reader with two USB ports.", "florence_label": "OCR"}, {"n": 69, "global_id": 18674, "domain": "Electronics", "asin": "B0053YJLL2", "title": "Lenovo Speaker M0520, Black ( 888010120 )", "image_url": "https://m.media-amazon.com/images/I/51ryJHoI3zL._AC_SL1000_.jpg", "florence_caption": "a pair of black speakers sitting on top of each other", "florence_label": "REDUNDANT"}, {"n": 70, "global_id": 24847, "domain": "Electronics", "asin": "B08DZSJ4MM", "title": "LG 43UN700T-B 43\" 4K UHD 3840x2160 IPS USB-C HDR 10 Monitor", "image_url": "https://m.media-amazon.com/images/I/71aVNmhmw-L._AC_SL1452_.jpg", "florence_caption": "A computer monitor with a monitor screen showing a video editing software.", "florence_label": "GENERIC"}, {"n": 71, "global_id": 55531, "domain": "Home_and_Kitchen", "asin": "B00D9HQ30W", "title": "FoodWorks Silicone Ice Pop Maker Molds/Popsicle Molds, Set of 6", "image_url": "https://m.media-amazon.com/images/I/412n2F9MECL._AC_.jpg", "florence_caption": "a group of colorful ice cream cones on a white background", "florence_label": "GENERIC"}, {"n": 72, "global_id": 64410, "domain": "Home_and_Kitchen", "asin": "B0055PU5DC", "title": "Tattler Reusable Wide Mouth Canning Lids & Rubber Rings-12/pkg", "image_url": "https://m.media-amazon.com/images/I/61PV63QzAkL._AC_SL1500_.jpg", "florence_caption": "a white plate with a red gasket on it", "florence_label": "GENERIC"}, {"n": 73, "global_id": 46941, "domain": "Home_and_Kitchen", "asin": "B071FB5KBQ", "title": "Glass Soap Pump Dispenser Floral and Butterfly 17 Ounce by A Ting", "image_url": "https://m.media-amazon.com/images/I/51xUXvHKy7L._AC_SL1000_.jpg", "florence_caption": "a soap dispenser with flowers on it", "florence_label": "REDUNDANT"}, {"n": 74, "global_id": 64772, "domain": "Home_and_Kitchen", "asin": "B008XLE0IQ", "title": "Extreme Freeze Reditainer 64 oz. Freezeable Deli Food Containers w/ Lids - Package of 8 - Food Storage", "image_url": "https://m.media-amazon.com/images/I/71QyC94iaVL._AC_SL1500_.jpg", "florence_caption": "A white plastic container with a lid on a white background.", "florence_label": "REDUNDANT"}, {"n": 75, "global_id": 47000, "domain": "Home_and_Kitchen", "asin": "B01KM5E0QW", "title": "Totally Bamboo Barkeeper's Salt Box, Margarita Salt Rimmer for Cocktail Drinks, Home Bar Accessory with Magnetic Swivel Lid", "image_url": "https://m.media-amazon.com/images/I/81N850r0hWL._AC_SL1500_.jpg", "florence_caption": "A wooden bowl filled with sea salt on top of a white background.", "florence_label": "REDUNDANT"}, {"n": 76, "global_id": 57078, "domain": "Home_and_Kitchen", "asin": "B014LPT0FA", "title": "Hamilton Beach Air Circulator, 16-Inch, Two Tone Gray", "image_url": "https://m.media-amazon.com/images/I/91MOtYDoXPL._AC_SL1500_.jpg", "florence_caption": "A gray fan sitting on top of a table.", "florence_label": "REDUNDANT"}, {"n": 77, "global_id": 36549, "domain": "Home_and_Kitchen", "asin": "B09N33VNWZ", "title": "KUHEE Air Mattress with Built in Pump, Queen air Mattress, Easy Inflation/Deflation,Comfortable Top Surface-Blow Up Airbed,Overheating Protection,80x60x18in,600lb MAX,Grey", "image_url": "https://m.media-amazon.com/images/I/515ZthbFzTL._AC_SL1500_.jpg", "florence_caption": "A woman laying on top of an air mattress.", "florence_label": "REDUNDANT"}, {"n": 78, "global_id": 52712, "domain": "Home_and_Kitchen", "asin": "B00VSM7BSI", "title": "Black Finish Wooden Cheval Bedroom Free Standing Floor Mirror", "image_url": "https://m.media-amazon.com/images/I/51USNOiGO7L._AC_SL1050_.jpg", "florence_caption": "A black standing mirror on a stand with a white background.", "florence_label": "REDUNDANT"}, {"n": 79, "global_id": 57313, "domain": "Home_and_Kitchen", "asin": "B01K1QP5YS", "title": "KINGWILLOW Woven Rectangular Wicker Storage Basket,Small Organizer Box", "image_url": "https://m.media-amazon.com/images/I/61103oqmM3L._AC_SL1000_.jpg", "florence_caption": "A white wicker basket with a floral pattern on it.", "florence_label": "ADDS"}, {"n": 80, "global_id": 33950, "domain": "Home_and_Kitchen", "asin": "B09WRQBB3T", "title": "Red Co. Glass Salt and Pepper Shaker Set in 5” Metal Carrying Toolbox Caddy with Wooden Handle", "image_url": "https://m.media-amazon.com/images/I/712j9VGlPaL._AC_SL1336_.jpg", "florence_caption": "a metal container with two salt and pepper shakers in it", "florence_label": "REDUNDANT"}, {"n": 81, "global_id": 77442, "domain": "Movies_and_TV", "asin": "B07MC5Z515", "title": "Motherless Brooklyn (Blu-ray + Digital)", "image_url": "https://m.media-amazon.com/images/I/91hZK3xOZzL._SL1500_.jpg", "florence_caption": "motherless brooklyn on blu - ray", "florence_label": "OCR"}, {"n": 82, "global_id": 85876, "domain": "Movies_and_TV", "asin": "076781326X", "title": "Oliver!", "image_url": "https://m.media-amazon.com/images/I/71tPBaDV9MS._SL1500_.jpg", "florence_caption": "A picture of a movie poster for Oliver.", "florence_label": "OCR"}, {"n": 83, "global_id": 87611, "domain": "Movies_and_TV", "asin": "B007PM207W", "title": "Patton / The Longest Day / The Sand Pebbles / Tora! Tora! Tora! [Blu-ray]", "image_url": "https://m.media-amazon.com/images/I/91Z7OnLsAUL._SL1500_.jpg", "florence_caption": "the longest day / tora tora / the sand pebbles", "florence_label": "OCR"}, {"n": 84, "global_id": 79788, "domain": "Movies_and_TV", "asin": "B01LTHWXXY", "title": "Deepwater Horizon [DVD]", "image_url": "https://m.media-amazon.com/images/I/51MnpyjVy9L.jpg", "florence_caption": "deepwater horizon dvd", "florence_label": "OCR"}, {"n": 85, "global_id": 83843, "domain": "Movies_and_TV", "asin": "B0045HCJ7G", "title": "What's Up, Doc? (DVD) (Rpkg)", "image_url": "https://m.media-amazon.com/images/I/51dSO7m4aIL.jpg", "florence_caption": "what's up doc? poster", "florence_label": "OCR"}, {"n": 86, "global_id": 80914, "domain": "Movies_and_TV", "asin": "B00008K76U", "title": "The Hunt for Red October", "image_url": "https://m.media-amazon.com/images/I/817gKCTGg4L._SL1426_.jpg", "florence_caption": "the hunt for red october", "florence_label": "OCR"}, {"n": 87, "global_id": 88326, "domain": "Movies_and_TV", "asin": "B00J8LXD60", "title": "Batman: Assault on Arkham (DVD)", "image_url": "https://m.media-amazon.com/images/I/91kjDpZOi1L._SL1500_.jpg", "florence_caption": "batman assault on arkham", "florence_label": "OCR"}, {"n": 88, "global_id": 87385, "domain": "Movies_and_TV", "asin": "B01FQT5JBO", "title": "I Am Wrath Digital", "image_url": "https://m.media-amazon.com/images/I/91fVYnyjVdL._SL1500_.jpg", "florence_caption": "i am wrath dvd", "florence_label": "OCR"}, {"n": 89, "global_id": 87953, "domain": "Movies_and_TV", "asin": "B077HP62ZQ", "title": "Justice League (BD)", "image_url": "https://m.media-amazon.com/images/I/91cPgKGdziL._SL1500_.jpg", "florence_caption": "justice league on blu - ray", "florence_label": "OCR"}, {"n": 90, "global_id": 87238, "domain": "Movies_and_TV", "asin": "B015TGNODE", "title": "The Intern", "image_url": "https://m.media-amazon.com/images/I/81ujZbQtSBL._SL1500_.jpg", "florence_caption": "A man and woman looking at a laptop computer.", "florence_label": "ADDS"}, {"n": 91, "global_id": 101941, "domain": "Tools_and_Home_Improvement", "asin": "B00B2QZEQG", "title": "Drill America - GLBCOX67/64 7/64\" x 6\" Cobalt Aircraft Extension Drill Bit, GLBCO Series", "image_url": "https://m.media-amazon.com/images/I/51dr7aakZgL._SL1280_.jpg", "florence_caption": "A drill bit on a white background.", "florence_label": "REDUNDANT"}, {"n": 92, "global_id": 101010, "domain": "Tools_and_Home_Improvement", "asin": "B005XPQ39Y", "title": "New Stens Air Filter 055-025 for Kohler 47 083 03-S1", "image_url": "https://m.media-amazon.com/images/I/61+dsifpmAL._AC_SL1000_.jpg", "florence_caption": "a air filter on a white background", "florence_label": "REDUNDANT"}, {"n": 93, "global_id": 105470, "domain": "Tools_and_Home_Improvement", "asin": "B00B21CHKW", "title": "Ansen Tools AN-269 6-Inch Soft Buffing and Polishing Wheel, 2-Piece", "image_url": "https://m.media-amazon.com/images/I/71S5zbmgF-L._AC_SL1500_.jpg", "florence_caption": "A pair of white and orange felt discs on a white background.", "florence_label": "ADDS"}, {"n": 94, "global_id": 88791, "domain": "Tools_and_Home_Improvement", "asin": "B000G7ZGNI", "title": "F Connector Installation & Removal Tool", "image_url": "https://m.media-amazon.com/images/I/61pTmEa0DYL._AC_SL1500_.jpg", "florence_caption": "a close up of a black connector on a white background", "florence_label": "REDUNDANT"}, {"n": 95, "global_id": 93754, "domain": "Tools_and_Home_Improvement", "asin": "B082X4CM8X", "title": "Goldblatt Folding Drywall Saw, Jab Saw, Hand Saw with Soft Grip Handle, Sheetrock Saw for Wallboard, Drywall, Plywood and PVC", "image_url": "https://m.media-amazon.com/images/I/61oZl8zoTSL._AC_SL1500_.jpg", "florence_caption": "a red saw with a black handle on a white background", "florence_label": "ADDS"}, {"n": 96, "global_id": 98532, "domain": "Tools_and_Home_Improvement", "asin": "B0852BJRXL", "title": "35-Pack Eye Protective Safety Cover Office Pack,Includes 25 Disposable Eye Shield Lenses per Pack and 10 Frames(Multicolor),Suitable for Meeting,Communication,Anti-Dropping", "image_url": "https://m.media-amazon.com/images/I/51kB74UxTzL._SL1105_.jpg", "florence_caption": "a pair of safety glasses with different colors", "florence_label": "ADDS"}, {"n": 97, "global_id": 89507, "domain": "Tools_and_Home_Improvement", "asin": "B08CP568HY", "title": "Klein Tools 69445 Magnetic Hanger Without Strap for Klein Tools Clamp Meters and Multimeters with Powerful Rare Earth Magnets", "image_url": "https://m.media-amazon.com/images/I/61xhBmfQ1SL._AC_SL1000_.jpg", "florence_caption": "klein tools 69445 c-1/2-inch x 1/4-inch hex key", "florence_label": "SUSPECT"}, {"n": 98, "global_id": 91948, "domain": "Tools_and_Home_Improvement", "asin": "B08BQSK2RF", "title": "Kreg KMA4100 Crosscut Station", "image_url": "https://m.media-amazon.com/images/I/81XcJo5GF7L._AC_SL1500_.jpg", "florence_caption": "A table saw with a piece of wood on top of it.", "florence_label": "REDUNDANT"}, {"n": 99, "global_id": 98374, "domain": "Tools_and_Home_Improvement", "asin": "B006T7A2CO", "title": "5/8\" Stainless Steel Wire Brush for Power Drill Impact Driver - Hex Shank By Protool", "image_url": "https://m.media-amazon.com/images/I/51BK-zLEWwL._AC_SL1000_.jpg", "florence_caption": "a wire brush on a white background", "florence_label": "REDUNDANT"}, {"n": 100, "global_id": 108731, "domain": "Tools_and_Home_Improvement", "asin": "B0B7XNWSFY", "title": "COOMYXIN Power Inverter Compatible with makita 18v Battery Turn DC 20V to AC 110V/150W Power Station", "image_url": "https://m.media-amazon.com/images/I/61kzIf6i+YL._AC_SL1500_.jpg", "florence_caption": "A power inverter and two batteries on a white background.", "florence_label": "REDUNDANT"}]''')
print(len(ITEMS), ITEMS[0])

## 1. Tải ảnh (có retry, lưu cache)

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0 (research pilot)"}
def fetch(item):
    path = IMG_DIR / f"{item['n']:03d}.jpg"
    if path.exists():
        return path, "cached"
    for attempt in range(4):
        try:
            r = requests.get(item["image_url"], headers=HEADERS, timeout=30)
            r.raise_for_status()
            img = Image.open(io.BytesIO(r.content)).convert("RGB")
            img.thumbnail((768, 768))  # giới hạn độ phân giải trước khi đưa vào model
            img.save(path, quality=92)
            return path, "ok"
        except Exception as e:
            err = f"{type(e).__name__}: {e}"
            time.sleep(1.5 * (attempt + 1))
    return None, err

for it in ITEMS:
    p, status = fetch(it)
    it["image_path"], it["image_status"] = (str(p) if p else None), status
print(pd.Series([it["image_status"] for it in ITEMS]).value_counts())

## 2. Nạp Qwen3-VL

In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, dtype=DTYPE, attn_implementation="sdpa", device_map={"": 0}
).eval()
devices = {str(p.device) for p in model.parameters()}
assert devices == {"cuda:0"}, f"model bị offload một phần: {devices}"
print(model.dtype, devices, f"{torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")

## 3. Ba prompt

In [ ]:
STRUCTURED = """You see a product image from an online store. The product title is: "{title}".
Describe ONLY visual information that is NOT already stated in the title.
Do not repeat or transcribe any words printed on the product, box, or cover.
Answer in at most 40 words as "key: value" pairs, choosing only relevant keys from:
color, material, shape/form factor, included items, art style, characters or scene shown, mood/theme, target audience cues, condition.
If the image clearly shows a different kind of product than the title, answer exactly: MISMATCH."""

GENERIC = "Describe this image in one sentence."

TITLE_ONLY = """The product title is: "{title}". You cannot see the product.
From your general knowledge only, guess its visual appearance.
Answer in at most 40 words as "key: value" pairs, choosing only relevant keys from:
color, material, shape/form factor, included items, art style, characters or scene shown, mood/theme, target audience cues, condition."""

@torch.inference_mode()
def generate(text, image_path=None):
    content = ([{"type": "image", "image": image_path}] if image_path else []) + [{"type": "text", "text": text}]
    messages = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new = out[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

# Smoke test trên item đầu tiên: output rỗng hoặc toàn ký tự lạ (fp16 tràn số) thì dừng lại ở đây.
_t = time.time()
_smoke = generate(STRUCTURED.format(title=ITEMS[0]["title"]), ITEMS[0]["image_path"])
print(f"smoke ({time.time()-_t:.1f}s):", repr(_smoke))
assert _smoke and re.search(r"[A-Za-z]{3}", _smoke), "output bất thường — kiểm tra dtype/transformers trước khi chạy tiếp"

## 4. Chạy (lưu dần, chạy lại cell sẽ tiếp tục từ chỗ dừng)

In [ ]:
RESULTS = OUT / f"results_{MODEL_ID.split('/')[-1]}.jsonl"
done = {json.loads(l)["n"] for l in RESULTS.read_text().splitlines()} if RESULTS.exists() else set()
t0 = time.time()
with RESULTS.open("a", encoding="utf-8") as f:
    for i, it in enumerate(ITEMS):
        if it["n"] in done:
            continue
        rec = {k: it[k] for k in ("n", "global_id", "domain", "asin", "title", "image_url", "florence_caption", "florence_label", "image_status")}
        rec["model"] = MODEL_ID
        if it["image_path"]:
            rec["structured"] = generate(STRUCTURED.format(title=it["title"]), it["image_path"])
            rec["generic"] = generate(GENERIC, it["image_path"])
        else:
            rec["structured"] = rec["generic"] = None
        rec["title_only"] = generate(TITLE_ONLY.format(title=it["title"]))
        f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
        if (i + 1) % 10 == 0:
            print(f"{i+1}/100  {time.time()-t0:.0f}s")
print("done", time.time() - t0, "s")

## 5. Chỉ số tự động

- `novel_words`: số từ nội dung trong caption **không** xuất hiện trong title.
- `title_overlap`: tỷ lệ từ nội dung của caption đã có trong title (cao ≈ chép lại title).
- `vision_vs_knowledge`: độ trùng (Jaccard) giữa `structured` và `title_only`. Cao nghĩa là thông tin có thể đoán được mà không cần ảnh.
- Item 19 và 65 đã biết là **ảnh gắn nhầm sản phẩm**: kỳ vọng `structured` trả `MISMATCH`.

In [ ]:
STOP = set("a an the of on in with and to for is are at by it its this that from as or be image picture photo shows showing product color material shape form factor included items art style characters scene shown mood theme target audience cues condition none n a".split())
tok = lambda s: set(re.findall(r"[a-z0-9]+", (s or "").lower())) - STOP
df = pd.DataFrame([json.loads(l) for l in RESULTS.read_text().splitlines()]).sort_values("n")
def overlap(c, t):
    c, t = tok(c), tok(t)
    return round(len(c & t) / len(c), 3) if c else None
for col in ["structured", "generic", "florence_caption"]:
    df[f"{col}_novel_words"] = [len(tok(c) - tok(t)) for c, t in zip(df[col], df["title"])]
    df[f"{col}_title_overlap"] = [overlap(c, t) for c, t in zip(df[col], df["title"])]
df["mismatch"] = df["structured"].fillna("").str.strip().str.upper().eq("MISMATCH")
df["vision_vs_knowledge"] = [
    round(len(tok(a) & tok(b)) / len(tok(a) | tok(b)), 3) if (tok(a) | tok(b)) else None
    for a, b in zip(df["structured"], df["title_only"])
]
summary = pd.DataFrame({
    "median_novel_words": [df[f"{c}_novel_words"].median() for c in ["florence_caption", "generic", "structured"]],
    "mean_title_overlap": [df[f"{c}_title_overlap"].mean() for c in ["florence_caption", "generic", "structured"]],
}, index=["florence (cũ)", "qwen generic", "qwen structured"])
display(summary.round(3))
print("MISMATCH:", int(df["mismatch"].sum()), "| item 19/65 (ảnh sai đã biết):", df.set_index("n").loc[[19, 65], "mismatch"].tolist())
print("vision_vs_knowledge median:", df["vision_vs_knowledge"].median())
print(df.groupby(df["domain"].eq("Video_Games").map({True: "Games", False: "Khác"}))[["structured_novel_words", "structured_title_overlap", "vision_vs_knowledge"]].median())

## 6. Bảng so sánh để gán nhãn thủ công

Điền cột `label_structured` bằng một trong: `OCR`, `REDUNDANT`, `ADDS`, `GENERIC`, `SUSPECT`, `MISMATCH_OK` (báo đúng ảnh sai).
Nhãn Florence cũ nằm ở `florence_label` để đối chiếu.

In [ ]:
pd.set_option("display.max_colwidth", 200)
view = df[["n", "domain", "title", "florence_caption", "florence_label", "generic", "structured", "title_only", "vision_vs_knowledge"]]
view.insert(7, "label_structured", "")
display(view)
view.to_csv(OUT / "label_sheet.csv", index=False)
df.to_json(OUT / "results_with_metrics.json", orient="records", force_ascii=False, indent=1)
print("saved:", OUT)

## 7. Tải kết quả về

In [ ]:
import shutil
archive = shutil.make_archive("/content/v10_qwen3vl_pilot_results", "zip", OUT, ".")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print(archive)